# Week 2 — decision extractor validation

This notebook validates the extraction pipeline for `llm-inference-batching-scheduler`. The fixed pilot contains Claude Fable 5 + Claude Code and Muse Spark 1.1 + mini-swe-agent. Gold-replay results below test the plumbing only and are not model accuracy.

In [1]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
OUT = ROOT / 'outputs' / '2026-08-23_week2_decision_extractor'

def read_jsonl(path):
    with Path(path).open(encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

print('Output folder:', OUT)

Output folder: d:\Studies\UZH-M.Sc\THESIS\design-decisions-by-LLMs\outputs\2026-08-23_week2_decision_extractor


## 1. Fixed pilot data

In [2]:
trajectories = read_jsonl(OUT / 'pilot_trajectories_two.jsonl')
gold = read_jsonl(OUT / 'pilot_gold_events_two.jsonl')
pilot_rows = []
for trajectory in trajectories:
    gold_record = next(item for item in gold if item['trial_id'] == trajectory['trial_id'])
    pilot_rows.append({
        'model': trajectory['model'],
        'harness': trajectory['harness'],
        'steps': len(trajectory['steps']),
        'gold events': len(gold_record['events']),
    })
pd.DataFrame(pilot_rows)

,model,harness,steps,gold events
0,Claude Fable 5,Claude Code,13,6
1,Muse Spark 1.1,mini-swe-agent,35,12


In [3]:
event_rows = [
    {'trial_id': record['trial_id'], **event}
    for record in gold for event in record['events']
]
events_df = pd.DataFrame(event_rows)
pd.crosstab(events_df['primary_category'], events_df['trial_id'], margins=True)

trial_id,30d171a7-5d7a-50b1-8d7d-31e8e8b6a734,c68f733c-d7d7-52fa-b063-e3b5cbd29eef,All
primary_category,,,
D1,4,2,6
D2,1,1,2
D4,1,1,2
D6,4,0,4
D7,1,1,2
D8,1,1,2
All,12,6,18


## 2. Plumbing test

In [4]:
replay_summary = json.loads((OUT / 'gold_replay_validation_summary.json').read_text(encoding='utf-8'))
pd.Series(replay_summary, name='gold replay — NOT model accuracy')

evaluator_version                        evaluator-v1
match_threshold                                   0.5
gold_events                                        18
predicted_events                                   18
true_positive_events                               18
false_positive_events                               0
missed_events                                       0
event_precision                                   1.0
event_recall                                      1.0
event_f1                                          1.0
mean_matched_step_overlap                         1.0
category_accuracy_on_matches                      1.0
reason_visibility_accuracy_on_matches             1.0
primary_reason_accuracy_on_matches                1.0
exact_boundary_accuracy_on_matches                1.0
prediction_records                                  2
invalid_prediction_records                          0
validation_error_counts                            {}
error_type_counts           

A perfect replay score is expected because the fixture is copied from gold. It confirms that the serializer, strict validator, event matcher, and metric calculations agree.

## 3. Genuine model run (when a provider is available)

In [5]:
live_summary_path = OUT / 'live_validation_summary.json'
if live_summary_path.exists():
    live_summary = json.loads(live_summary_path.read_text(encoding='utf-8'))
    metric_keys = [
        'event_precision', 'event_recall', 'event_f1',
        'category_accuracy_on_matches',
        'reason_visibility_accuracy_on_matches',
        'primary_reason_accuracy_on_matches',
        'exact_boundary_accuracy_on_matches',
    ]
    metric_values = [live_summary.get(key) for key in metric_keys]
    ax = pd.Series(metric_values, index=metric_keys).plot.bar(figsize=(10, 4), ylim=(0, 1), color='#4c956c')
    ax.set_ylabel('Score')
    ax.set_title('Live extractor compared with gold')
    plt.xticks(rotation=35, ha='right')
    plt.tight_layout()
else:
    failed_path = OUT / 'pilot_prediction_claude_fable.jsonl'
    failed = read_jsonl(failed_path) if failed_path.exists() else []
    print('No live validation summary yet. Provider status:')
    display(pd.DataFrame([{
        'trial_id': item['trial_id'],
        'valid': item['valid'],
        'error': item['errors'][0]['message'] if item.get('errors') else None,
    } for item in failed]))

No live validation summary yet. Provider status:


,trial_id,valid,error
0,c68f733c-d7d7-52fa-b063-e3b5cbd29eef,False,RuntimeError: Provider command failed: exit co...


## 4. Error analysis

In [6]:
error_path = OUT / 'live_error_analysis.csv'
if error_path.exists():
    errors_df = pd.read_csv(error_path)
    display(errors_df)
    display(errors_df['error_type'].value_counts())
else:
    print('Run a working live provider and the evaluator to create the error table.')

Run a working live provider and the evaluator to create the error table.


## Findings

- The Week 2 data path is fixed and reproducible: 2 pilot trajectories, 48 total gold events across all 6 trajectories, and 18 gold events in the pilot.
- Exact-substring evidence validation prevents the extractor from quoting text that is absent from the trajectory.
- Decision detection and label quality are measured separately. An event can match by steps while still having a wrong category, reason, or boundary.
- Gold replay validates the pipeline but says nothing about LLM accuracy.
- A genuine quality result requires a working authenticated provider; a provider failure must be reported separately from extraction errors.